# CNEFE Data Cleanin

## Setup

In [1]:
from pyspark.sql import SparkSession
from pathlib import Path
import json
import os 

PROJECT_PATH = Path.home() / "projects" / "brazilian_address_linkage"
os.chdir(PROJECT_PATH)

INCOSISTENT_VALS_MAP = json.load(open("config/inconsistent_annotations.json", "r"))
spark = (
                SparkSession
                .builder
                .appName("CNEFE Data Cleaning")
                .config("spark.sql.execution.arrow.pyspark.enabled", "true")
                .config("spark.drive.memory", "10g")
                .config("spark.executor.memory", "10g")
                .config("spark.executor.cores", "4")
                .getOrCreate()
)
cnefe_raw_df = spark.read.parquet(str(PROJECT_PATH / "data/integrated/integrated_cnefe_addresses.parquet") )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/03 07:55:16 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/09/03 07:55:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/03 07:55:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


 ## Clean Inconsistent Values 

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import functions as F, DataFrame

def display_missingness(df: DataFrame) -> None:
    total_rows = df.count()

    fulfillment_exprs = [
        F.struct(
            F.lit(c).alias("column"),
            (F.count(F.col(c)) / F.lit(total_rows) * 100).alias("fulfillment_pct"),
        )
        for c in df.columns
    ]

    result = (
        df.select(F.array(*fulfillment_exprs).alias("stats"))
        .selectExpr("explode(stats) as stats")
        .select("stats.column", "stats.fulfillment_pct")
    )

    result.orderBy(F.col("fulfillment_pct").desc()).show(
        len(df.columns), truncate=False
    )
    

    
def is_undefined(col_name: str) -> F.Column:
    undefined_expressions = INCOSISTENT_VALS_MAP.get(col_name, [])
    if not undefined_expressions:
        return F.lit(False)
    return F.col(col_name).cast("string").isin(*undefined_expressions)

In [3]:
import pandas as pd

total_rows = cnefe_raw_df.count()
categorical_cols = [
    f.name for f in cnefe_raw_df.schema.fields if str(f.dataType) == "StringType()"
]
undefined_count=[]
for col_name in categorical_cols:
    print(f"Column: {col_name}")
    count_undefined_df = cnefe_raw_df.filter(is_undefined(col_name)) \
      .groupBy(F.col(col_name)) \
      .count().alias("count") \
      .orderBy(F.col("count").desc())
    
    if count_undefined_df.count() > 0:
        total_undefined_count = count_undefined_df.agg(F.sum("count")).collect()[0][0]
        percentage = round((total_undefined_count / total_rows) * 100, 3)
        undefined_count.append({"column": col_name, "count": total_undefined_count, "pct_db": percentage})

pd.DataFrame(undefined_count)
        

Column: COD_SETOR
Column: CEP
Column: DSC_LOCALIDADE


Column: NOM_TIPO_SEGLOGR


Column: NOM_TITULO_SEGLOGR
Column: NOM_SEGLOGR


Column: DSC_MODIFICADOR
Column: NOM_COMP_ELEM1
Column: VAL_COMP_ELEM1
Column: NOM_COMP_ELEM2
Column: VAL_COMP_ELEM2
Column: NOM_COMP_ELEM3
Column: VAL_COMP_ELEM3
Column: NOM_COMP_ELEM4
Column: VAL_COMP_ELEM4
Column: NOM_COMP_ELEM5
Column: VAL_COMP_ELEM5
Column: DSC_ESTABELECIMENTO
Column: COD_INDICADOR_ESTAB_ENDERECO
Column: COD_INDICADOR_CONST_ENDERECO
Column: COD_INDICADOR_FINALIDADE_CONST
Column: UF


,column,count,pct_db
0,DSC_LOCALIDADE,7089,0.006
1,NOM_TIPO_SEGLOGR,28041,0.025
2,NOM_SEGLOGR,1236474,1.113


In [5]:
raw_count = cnefe_raw_df.count()
1236474/raw_count * 100

1.1129090943866213

In [6]:
from pyspark.sql import functions as F

columns_names = list(INCOSISTENT_VALS_MAP.keys())

cnefe_cleaned_df = cnefe_raw_df
for c in columns_names:
    cond = is_undefined(c)
    n_bad = cnefe_cleaned_df.filter(cond).count()
    print(f"Detected undefined expressions in column: {c} -> {n_bad}")
    cnefe_cleaned_df = cnefe_cleaned_df.withColumn(
        c, F.when(cond, None).otherwise(F.col(c))
    )

display_missingness(cnefe_cleaned_df.select(columns_names))

Detected undefined expressions in column: DSC_LOCALIDADE -> 7089
Detected undefined expressions in column: CEP -> 0
Detected undefined expressions in column: COD_TIPO_ESPECI -> 0


Detected undefined expressions in column: NOM_SEGLOGR -> 1236474
Detected undefined expressions in column: NOM_TIPO_SEGLOGR -> 28041


+----------------+-----------------+
|column          |fulfillment_pct  |
+----------------+-----------------+
|CEP             |100.0            |
|DSC_LOCALIDADE  |99.99361582677317|
|NOM_TIPO_SEGLOGR|99.97476122917611|
|NOM_SEGLOGR     |98.88709090561338|
|COD_TIPO_ESPECI |81.6401753779999 |
+----------------+-----------------+



## Normalizing Addresses